# Qwen2-VL-2B 纯 OCR 微调
点左边播放按钮 ▶ 一步步跑就行

In [ ]:
# ① 上传 ocr_data.zip
from google.colab import files
import zipfile, os, shutil
print("请选择 ocr_data.zip 上传：")
uploaded = files.upload()

os.makedirs('data/ocr', exist_ok=True)
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('data/ocr/')
        # 如果解压后有 data/ocr/data/ocr/ 嵌套，拍平
        if os.path.exists('data/ocr/data/ocr'):
            for item in os.listdir('data/ocr/data/ocr'):
                src = os.path.join('data/ocr/data/ocr', item)
                dst = os.path.join('data/ocr', item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree('data/ocr/data')
        # 如果是 ocr/images, ocr/labels.jsonl 这种，拍平
        if os.path.exists('data/ocr/ocr'):
            for item in os.listdir('data/ocr/ocr'):
                src = os.path.join('data/ocr/ocr', item)
                dst = os.path.join('data/ocr', item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree('data/ocr/ocr', ignore_errors=True)
        print('解压完成')

# 验证
if os.path.exists('data/ocr/labels.jsonl'):
    with open('data/ocr/labels.jsonl') as f:
        n = sum(1 for _ in f)
    print(f'共 {n} 条训练数据')
else:
    # 找一下 labels.jsonl 到底在哪
    print('文件结构：')
    for root, dirs, files in os.walk('data'):
        for f in files:
            print(f'  {os.path.join(root, f)}')

In [ ]:
# ② 装依赖（约 2 分钟）
!pip install -q transformers pillow accelerate
!pip install -q torchao --upgrade
!pip install -q peft huggingface_hub
import os; os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print('依赖装完')

In [ ]:
# ③ 加载模型 + LoRA
import torch, json, os
from torch.utils.data import Dataset, Subset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from PIL import Image

print('下载 Qwen2-VL-2B 模型中（约 4GB，3-5 分钟）...')
model = Qwen2VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2-VL-2B-Instruct',
    torch_dtype=torch.float16, device_map='auto')
processor = AutoProcessor.from_pretrained('Qwen/Qwen2-VL-2B-Instruct')

for p in model.model.visual.parameters():
    p.requires_grad = False

lora = LoraConfig(r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    lora_dropout=0.1, bias='none', task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print('模型加载完')

In [ ]:
# ④ 加载数据
class OCRDataset(Dataset):
    def __init__(self, path):
        with open(path) as f:
            self.data = [json.loads(l) for l in f if l.strip()]
        self.base = os.path.dirname(path)
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        d = self.data[i]
        img = Image.open(os.path.join(self.base, d['image'])).convert('RGB')
        return {'image': img, 'text': d['text']}

ds = OCRDataset('data/ocr/labels.jsonl')
s = int(len(ds) * 0.8)
train_ds = Subset(ds, range(s))
val_ds   = Subset(ds, range(s, len(ds)))
print(f'训练 {len(train_ds)} 条, 验证 {len(val_ds)} 条')

In [ ]:
# ⑤ 开始训练（约 10-15 分钟）
def collate_fn(examples):
    texts, images = [], []
    for item in examples:
        msgs = [{'role': 'user', 'content': [
            {'type': 'image', 'image': item['image']},
            {'type': 'text', 'text': '只输出图片中的文字，不要加解释。'}]},
            {'role': 'assistant', 'content': item['text']}]
        texts.append(processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
        images.append(item['image'])
    inputs = processor(text=texts, images=images, return_tensors='pt', padding=True)
    inputs['labels'] = inputs['input_ids'].clone()
    return inputs

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='./ckpt', num_train_epochs=3,
        per_device_train_batch_size=4, learning_rate=2e-4,
        fp16=True, logging_steps=10, save_strategy='epoch',
        eval_strategy='epoch', report_to='none', remove_unused_columns=False),
    train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate_fn)
trainer.train()
print('训练完成')

In [ ]:
# ⑥ 合并并下载
from google.colab import files
merged = model.merge_and_unload()
merged.save_pretrained('./Qwen2-VL-2B-OCR')
processor.save_pretrained('./Qwen2-VL-2B-OCR')
!zip -r Qwen2-VL-2B-OCR.zip Qwen2-VL-2B-OCR/
print('下载 Qwen2-VL-2B-OCR.zip，解压后放 RK3588 model/ 目录')
files.download('Qwen2-VL-2B-OCR.zip')